# Login / Logout 로그 SQL 생성
- 로그인 → 로그아웃 쌍으로 생성
- user_login_id: user0001 ~ user0100, user_id: 1 ~ 100 매칭
- client_uuid: 세션마다 고유 UUID (비로그인 사용자 구분용)
- 타임스탬프: login < logout 순서 보장

In [1]:
import random
import json
import uuid
from datetime import datetime, timedelta

In [2]:
# ───────────────────────────────────────────
# 설정값
# ───────────────────────────────────────────
START_DATE  = datetime(2026, 1, 1, 0, 0, 0)
END_DATE    = datetime(2026, 6, 17, 23, 59, 59)
PAIR_COUNT  = 250   # 생성할 login/logout 쌍 수 (각 이벤트 수 = PAIR_COUNT)

# 세션 체류 시간: 최소 1분 ~ 최대 120분
SESSION_MIN_SEC = 60
SESSION_MAX_SEC = 60 * 120

In [3]:
def random_datetime(start, end):
    delta = end - start
    random_seconds = random.randint(0, int(delta.total_seconds()))
    return start + timedelta(seconds=random_seconds)

def format_kst(dt):
    """event_timestamp 포맷 (KST +09:00)"""
    return dt.strftime('%Y-%m-%dT%H:%M:%S.') + f"{dt.microsecond // 1000:03d}+09:00"

def format_history_ts(dt):
    """history_timestamp 포맷 (마이크로초 포함)"""
    return dt.strftime('%Y-%m-%d %H:%M:%S.%f')

In [4]:
login_rows  = []
logout_rows = []

for _ in range(PAIR_COUNT):
    # 유저 선택 (user_id 1~100, user_login_id 매칭)
    user_id       = random.randint(1, 100)
    user_login_id = f'user{user_id:04d}'

    # 세션마다 고유 UUID
    client_uuid = str(uuid.uuid4())

    # ── LOGIN 타임스탬프 ──
    # event_timestamp 기준으로 먼저 뽑고, history_timestamp = event_ts - 1초
    login_event_ts   = random_datetime(START_DATE, END_DATE)
    login_history_ts = login_event_ts - timedelta(seconds=1)

    # ── LOGOUT 타임스탬프 ──
    # 세션 체류 시간만큼 뒤
    session_duration  = random.randint(SESSION_MIN_SEC, SESSION_MAX_SEC)
    logout_event_ts   = login_event_ts + timedelta(seconds=session_duration)
    logout_history_ts = logout_event_ts - timedelta(seconds=1)

    # END_DATE 초과 방지
    if logout_event_ts > END_DATE:
        logout_event_ts   = END_DATE
        logout_history_ts = END_DATE - timedelta(seconds=1)

    # ── LOGIN JSON ──
    login_json = json.dumps({
        'event_name':      'login',
        'user_id':         user_id,
        'user_login_id':   user_login_id,
        'client_uuid':     client_uuid,
        'event_timestamp': format_kst(login_event_ts)
    }, ensure_ascii=False)

    # ── LOGOUT JSON ──
    logout_json = json.dumps({
        'event_name':      'logout',
        'user_id':         user_id,
        'user_login_id':   user_login_id,
        'client_uuid':     client_uuid,
        'event_timestamp': format_kst(logout_event_ts)
    }, ensure_ascii=False)

    login_rows.append((login_history_ts, login_json))
    logout_rows.append((logout_history_ts, logout_json))

print(f'✅ {PAIR_COUNT}쌍 생성 완료 (login {len(login_rows)}건 / logout {len(logout_rows)}건)')

✅ 250쌍 생성 완료 (login 250건 / logout 250건)


In [5]:
def build_sql(table_name, rows):
    lines  = [f"INSERT INTO {table_name} (history_timestamp, json_log) VALUES"]
    values = []
    for history_ts, json_log in rows:
        ts_str  = format_history_ts(history_ts)
        escaped = json_log.replace("'", "''")
        values.append(f"  ('{ts_str}', '{escaped}')")
    lines.append(',\n'.join(values) + ';')
    return '\n'.join(lines)

login_sql  = build_sql('login_history',  login_rows)
logout_sql = build_sql('logout_history', logout_rows)

with open('login_logs.sql',  'w', encoding='utf-8') as f:
    f.write(login_sql)

with open('logout_logs.sql', 'w', encoding='utf-8') as f:
    f.write(logout_sql)

print('✅ login_logs.sql  저장 완료')
print('✅ logout_logs.sql 저장 완료')

✅ login_logs.sql  저장 완료
✅ logout_logs.sql 저장 완료


In [6]:
# ── 미리보기 ──
print('=== LOGIN SQL (앞 500자) ===')
print(login_sql[:500])
print()
print('=== LOGOUT SQL (앞 500자) ===')
print(logout_sql[:500])

=== LOGIN SQL (앞 500자) ===
INSERT INTO login_history (history_timestamp, json_log) VALUES
  ('2026-03-23 02:14:49.000000', '{"event_name": "login", "user_id": 87, "user_login_id": "user0087", "client_uuid": "77a44411-d267-44d3-b163-1fc3439cab09", "event_timestamp": "2026-03-23T02:14:50.000+09:00"}'),
  ('2026-04-13 07:37:26.000000', '{"event_name": "login", "user_id": 58, "user_login_id": "user0058", "client_uuid": "ff2df99c-5605-4631-906a-fda77dce023a", "event_timestamp": "2026-04-13T07:37:27.000+09:00"}'),
  ('2026-03-2

=== LOGOUT SQL (앞 500자) ===
INSERT INTO logout_history (history_timestamp, json_log) VALUES
  ('2026-03-23 03:34:43.000000', '{"event_name": "logout", "user_id": 87, "user_login_id": "user0087", "client_uuid": "77a44411-d267-44d3-b163-1fc3439cab09", "event_timestamp": "2026-03-23T03:34:44.000+09:00"}'),
  ('2026-04-13 08:25:58.000000', '{"event_name": "logout", "user_id": 58, "user_login_id": "user0058", "client_uuid": "ff2df99c-5605-4631-906a-fda77dce023a", "event_t